In [1]:
# ================================
# Import libraries
# ================================
# Standard library
import glob
import json
import os
import shutil
from pathlib import Path

# Third-party library
import numpy as np
from PIL import Image
from langchain_community.vectorstores import Chroma
from langchain_core.documents import Document
from langchain_aws import BedrockEmbeddings

print("✅ Environment ready")
print(os.getcwd())           # current directory
print(os.access(os.getcwd(), os.W_OK))  # is it writable?

✅ Environment ready
/Users/anirudh/Desktop/Courseera/IBM-RAG and agentic AI/RAG and Agentic AI Capstone Project
True


In [2]:
# ================================
# Download and prepare image data
# ================================
ZIP_URL = "https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/5_Rr6ohviItzucyWk6nkrw/synthetic-recipe-images.zip"
ZIP_PATH = "synthetic-recipe-images.zip"
IMG_DIR  = "recipe_images"

!wget -q -O {ZIP_PATH} {ZIP_URL}
!unzip -oq {ZIP_PATH} -d {IMG_DIR}

image_paths = sorted(glob.glob(f"{IMG_DIR}/**/*.png", recursive=True))
print(f"✅ Images found: {len(image_paths)}")


✅ Images found: 109


In [3]:
# ================================
# Load structured data from Module 1
# ================================

# NOTE: Ensure the JSON files are in your current working directory.

with open("structured_restaurant_data.json", "r") as f:
    restaurants = json.load(f)

with open("augmented_food_recipe.json", "r") as f:
    recipes = json.load(f)

print(f"✅ Loaded restaurants: {len(restaurants)}")
print(f"✅ Loaded recipes:     {len(recipes)}")


✅ Loaded restaurants: 210
✅ Loaded recipes:     109


In [4]:
# ================================
# Initialize embedding models
# ================================
import base64
import numpy as np
from langchain_aws import BedrockEmbeddings
import boto3

# ---- Text embedding model (1024-d) ----
text_embedder = BedrockEmbeddings(
    model_id="amazon.titan-embed-text-v2:0",
    region_name="us-east-1",
    model_kwargs={"normalize": True}  # cosine-ready
)

def embed_texts(texts, batch_size=64):
    embeddings = []
    for i in range(0, len(texts), batch_size):
        batch = texts[i:i+batch_size]
        batch_embeddings = text_embedder.embed_documents(batch)
        embeddings.extend(batch_embeddings)
    return np.array(embeddings, dtype=np.float32)

print("✅ Text embedder ready")

bedrock_client = boto3.client(
    service_name="bedrock-runtime",
    region_name="us-east-1"
)

def embed_images(paths, batch_size=16):
    embeddings = []
    for i in range(0, len(paths), batch_size):
        batch = paths[i:i+batch_size]
        for image_path in batch:
            # Encode image to base64
            with open(image_path, 'rb') as f:
                image_data = base64.b64encode(f.read()).decode('utf-8')

            # Call Titan Multimodal directly via boto3
            body = json.dumps({
                "inputImage": image_data,  # base64 encoded image
                "embeddingConfig": {
                    "outputEmbeddingLength": 1024
                }
            })

            response = bedrock_client.invoke_model(
                modelId="amazon.titan-embed-image-v1",
                body=body,
                contentType="application/json",
                accept="application/json"
            )

            result = json.loads(response['body'].read())
            embedding = result['embedding']

            # Normalize - cosine ready
            vector = np.array(embedding, dtype=np.float32)
            vector = vector / np.linalg.norm(vector)
            embeddings.append(vector)

    return np.array(embeddings, dtype=np.float32)

print("✅ Image embedder ready")

✅ Text embedder ready
✅ Image embedder ready


In [5]:
# ================================
# Build article and image documents
# ================================

# -------- articles --------
article_docs = []

for i, r in enumerate(restaurants):
    name = str(r.get("name", "")).strip()
    if not name:
        continue

    text = (
    f"Restaurant: {name}\n"
    f"Cuisine: {r.get('food_style','')}\n"
    f"Location: {r.get('location','')}"
    )

    # GUARANTEED UNIQUE
    doc_id = f"rest_{i}"

    article_docs.append(
        Document(
            page_content=text.strip(),
            metadata={
                "doc_id": doc_id,
                "cuisine": r.get("food_style"),
                "location": r.get("location"),
                "source": "restaurant",
            },
        )
    )

print("✅ article docs:", len(article_docs))


# -------- images --------
image_docs = []

for i, (p, rec) in enumerate(zip(image_paths, recipes)):
    doc_id = f"img_{i}"

    image_docs.append(
        Document(
            # keeps retrieval results readable
            page_content=rec.get("name", f"recipe image {i}"),
            metadata={
                "doc_id": doc_id,
                "image_path": p,
                "source": "recipe_image",
                "recipe_id": rec.get("id"),
                "cuisine": rec.get("cuisine"),
            },
        )
    )

print("✅ image docs:", len(image_docs))
print(article_docs[0].metadata)
print(image_docs[0].metadata)

✅ article docs: 210
✅ image docs: 109
{'doc_id': 'rest_0', 'cuisine': 'Farm-to-Table Californian', 'location': 'Silver Lake', 'source': 'restaurant'}
{'doc_id': 'img_0', 'image_path': 'recipe_images/synthetic_recipe_images/recipe1.png', 'source': 'recipe_image', 'recipe_id': 1, 'cuisine': 'Italian'}


In [6]:
DB_DIR = str((Path.cwd() / "chroma_multimodal").resolve())
if os.path.isdir(DB_DIR):
    shutil.rmtree(DB_DIR)

# ----- article DB -----
A = embed_texts([d.page_content for d in article_docs])
article_db = Chroma(
    collection_name="restaurant_articles",
    persist_directory=DB_DIR,
)
article_db._collection.upsert(
    ids=[d.metadata["doc_id"] for d in article_docs],  # ✅ 'rest_0', 'rest_1'...
    embeddings=A.tolist(),
    documents=[d.page_content for d in article_docs],
    metadatas=[d.metadata for d in article_docs],
)
print("✅ Article DB ready")

# ----- image DB -----
V = embed_images([d.metadata["image_path"] for d in image_docs])
image_db = Chroma(
    collection_name="food_images",
    persist_directory=DB_DIR,
)
image_db._collection.upsert(
    ids=[d.metadata["doc_id"] for d in image_docs],  # ✅ 'img_0', 'img_1'...
    embeddings=V.tolist(),
    documents=[d.page_content for d in image_docs],
    metadatas=[d.metadata for d in image_docs],
)
print("✅ Image DB ready")
print("🎉 Multimodal Vector Index Construction COMPLETE")

/var/folders/kv/9b0l8lv56tsc3yvfy9l9nq180000gn/T/ipykernel_37919/3050389829.py:7: LangChainDeprecationWarning: The class `Chroma` was deprecated in LangChain 0.2.9 and will be removed in 1.0. An updated version of the class exists in the `langchain-chroma package and should be used instead. To use it run `pip install -U `langchain-chroma` and import as `from `langchain_chroma import Chroma``.
  article_db = Chroma(


✅ Article DB ready
✅ Image DB ready
🎉 Multimodal Vector Index Construction COMPLETE


In [7]:
import chromadb

# Connect to existing DB
client = chromadb.PersistentClient(path=DB_DIR)

# List all collections
collections = client.list_collections()
print("Collections in DB:")
for col in collections:
    print(f"  - {col.name} | Count: {col.count()}")

Collections in DB:
  - food_images | Count: 109
  - restaurant_articles | Count: 210
